# 🎙️ STT EVAL — Arabic Speech-to-Text Evaluation & Robustness Notebook


This notebook provides an end-to-end evaluation and stress-testing pipeline for Arabic STT models (Whisper variants, fine-tuned models, and custom architectures).

### Key Capabilities:
- 🎯 **Dialectal Arabic Support**: Evaluates across Arab dialects using the `oddadmix/dialectal-arabic-lahgtna-v2` streaming dataset.
- 📁 **Flexible Ingestion**: Stream from Hugging Face, upload `.zip`, upload individual files, connect Google Drive, or use Google FLEURS.
- 🔊 **Acoustic Robustness Suite**: Multi-level White Noise (20dB, 10dB, 5dB), Telephony compression (300-3400Hz, 8kHz, $\mu$-law), Speed (0.85x, 1.15x), Gain (-25dB), and Room Reverb (RT60=0.5s).
- 📊 **Deep Diagnostics**: WER/CER per dialect, S/D/I rates, 95% Bootstrap CI, RTF throughput, failure flags (hallucinations & repetition loops).
- 🎧 **Embedded Audio Players**: Listen to worst-case errors and compare clean vs. corrupted audio directly in the notebook.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚙️ Step 0: Install Dependencies & Setup Environment
# ═══════════════════════════════════════════════════════════════

!pip install -q transformers accelerate torch torchaudio librosa soundfile jiwer datasets tqdm pandas matplotlib scipy

import os, sys, json, time, re, tempfile, zipfile
from itertools import islice
from pathlib import Path
from dataclasses import asdict, dataclass
from typing import Callable, List, Optional, Sequence, Union
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Audio, HTML
import warnings
warnings.filterwarnings("ignore")

import jiwer
import librosa
from scipy.signal import butter, fftconvolve, sosfilt
from tqdm.auto import tqdm

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found, running on CPU (slow)")
print("Setup done")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 Core Evaluation Engine & Audio Perturbations
# ═══════════════════════════════════════════════════════════════

SR = 16000

# ────── Arabic Normalization ──────
_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
_TATWEEL = "\u0640"
_ALEF_FORMS = re.compile("[إأآٱ]")
_NON_WORD = re.compile(r"[^\w\s]")
_SPACES = re.compile(r"\s+")
_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
_TAGS = re.compile(r"<[^>]*>|\[[^\]]*\]|\([^\)]*\)|<[^>]*$|\[[^\]]*$|\([^\)]*$")

@dataclass(frozen=True)
class NormConfig:
    remove_diacritics: bool = True
    remove_tags: bool = True
    unify_alef: bool = True
    unify_ya: bool = True
    unify_ta_marbuta: bool = True
    unify_hamza_seat: bool = False
    map_digits: bool = True
    lowercase_latin: bool = True
    def describe(self):
        return asdict(self)

# normalize Arabic text
def normalize_arabic(text, cfg=NormConfig()):
    text = str(text or "").strip()
    if cfg.remove_tags:
        text = _TAGS.sub(" ", text)
    if cfg.remove_diacritics:
        text = _DIACRITICS.sub("", text)
    text = text.replace(_TATWEEL, "")
    if cfg.unify_alef:
        text = _ALEF_FORMS.sub("ا", text)
    if cfg.unify_ya:
        text = text.replace("ى", "ي")
    if cfg.unify_ta_marbuta:
        text = text.replace("ة", "ه")
    if cfg.unify_hamza_seat:
        text = text.replace("ؤ", "و").replace("ئ", "ي")
    if cfg.map_digits:
        text = text.translate(_DIGITS)
    if cfg.lowercase_latin:
        text = text.lower()
    text = _NON_WORD.sub(" ", text)
    return _SPACES.sub(" ", text).strip()

# per-utterance error counts
def utterance_stats(ref, hyp):
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return {"w_sub": w.substitutions, "w_del": w.deletions, "w_ins": w.insertions, "w_hit": w.hits,
            "c_sub": c.substitutions, "c_del": c.deletions, "c_ins": c.insertions, "c_hit": c.hits}

def score_utterances(refs, hyps):
    rows = [{"ref": r, "hyp": h, **utterance_stats(r, h)} for r, h in zip(refs, hyps) if r]
    df = pd.DataFrame(rows)
    df["w_err"] = df.w_sub + df.w_del + df.w_ins
    df["w_ref_len"] = df.w_sub + df.w_del + df.w_hit
    df["c_err"] = df.c_sub + df.c_del + df.c_ins
    df["c_ref_len"] = df.c_sub + df.c_del + df.c_hit
    df["wer"] = df.w_err / df.w_ref_len
    df["cer"] = df.c_err / df.c_ref_len
    return df

def corpus_rate(df, unit="w"):
    return float(df[f"{unit}_err"].sum() / df[f"{unit}_ref_len"].sum())

# bootstrap confidence interval
def bootstrap_ci(df, unit="w", n_boot=1000, seed=42, alpha=0.05):
    err, length = df[f"{unit}_err"].to_numpy(), df[f"{unit}_ref_len"].to_numpy()
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    rates = err[idx].sum(axis=1) / length[idx].sum(axis=1)
    lo, hi = np.percentile(rates, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

def summarize(df, n_boot=1000, seed=42):
    ref_words = df.w_ref_len.sum()
    lo, hi = bootstrap_ci(df, "w", n_boot, seed)
    return {"n_scored": len(df), "wer": corpus_rate(df, "w"), "wer_ci_low": lo, "wer_ci_high": hi,
            "cer": corpus_rate(df, "c"), "sub_rate": df.w_sub.sum() / ref_words,
            "del_rate": df.w_del.sum() / ref_words, "ins_rate": df.w_ins.sum() / ref_words}

# most frequent word errors
def top_confusions(refs, hyps, k=25):
    counts = Counter()
    for ref, hyp in zip(refs, hyps):
        if not ref: continue
        out = jiwer.process_words(ref, hyp)
        r, h = ref.split(), hyp.split()
        for chunk in out.alignments[0]:
            rs, re_, hs, he = chunk.ref_start_idx, chunk.ref_end_idx, chunk.hyp_start_idx, chunk.hyp_end_idx
            if chunk.type == "substitute":
                counts.update(("sub", a, b) for a, b in zip(r[rs:re_], h[hs:he]))
            elif chunk.type == "delete":
                counts.update(("del", a, "") for a in r[rs:re_])
            elif chunk.type == "insert":
                counts.update(("ins", "", b) for b in h[hs:he])
    rows = [{"type": t, "reference": a, "prediction": b, "count": n} for (t, a, b), n in counts.most_common(k)]
    return pd.DataFrame(rows, columns=["type", "reference", "prediction", "count"])

def _dbfs(value):
    return float(20 * np.log10(max(float(value), 1e-12)))

def _rms(y):
    return float(np.sqrt(np.mean(np.square(y, dtype=np.float64)) + 1e-12))

# basic audio stats
def audio_features(y, sr=SR):
    y = np.asarray(y, dtype=np.float32)
    if y.size == 0:
        return {"duration_sec": 0.0, "rms_dbfs": -120.0, "peak_dbfs": -120.0,
                "clipping_ratio": 0.0, "silence_ratio": 1.0, "noise_floor_dbfs_proxy": -120.0}
    frame, hop = int(sr * 0.025), int(sr * 0.010)
    rms = _rms(y)
    frames = librosa.util.frame(y, frame_length=frame, hop_length=hop) if len(y) >= frame else y.reshape(-1, 1)
    frame_rms = np.sqrt(np.mean(np.square(frames, dtype=np.float64), axis=0))
    quiet = frame_rms[frame_rms <= np.percentile(frame_rms, 20)]
    return {"duration_sec": len(y) / sr, "rms_dbfs": _dbfs(rms), "peak_dbfs": _dbfs(np.max(np.abs(y))),
            "clipping_ratio": float(np.mean(np.abs(y) >= 0.999)),
            "silence_ratio": float(np.mean(frame_rms < max(1e-4, rms * 0.03))),
            "noise_floor_dbfs_proxy": _dbfs(np.median(quiet))}

_ARABIC_LETTER = re.compile(r"[\u0621-\u064A]")
_ANY_LETTER = re.compile(r"[^\W\d_]")

def max_repeat(tokens, max_ngram=4):
    best = 1 if tokens else 0
    for n in range(1, max_ngram + 1):
        for i in range(len(tokens) - n + 1):
            gram, reps, j = tokens[i:i + n], 1, i + n
            while tokens[j:j + n] == gram:
                reps += 1; j += n
            best = max(best, reps)
    return best

def non_arabic_ratio(text):
    letters = _ANY_LETTER.findall(text)
    if not letters: return 0.0
    return 1 - len(_ARABIC_LETTER.findall(text)) / len(letters)

# flag empty, long, repeated, foreign output
def failure_flags(ref, hyp, length_ratio_max=2.0, repeat_min=3, foreign_max=0.3):
    ref_len, hyp_words = len(ref.split()), hyp.split()
    ratio = len(hyp_words) / ref_len if ref_len else float("inf")
    return {"flag_empty": not hyp.strip(), "flag_too_long": ratio > length_ratio_max,
            "flag_repetition": max_repeat(hyp_words) >= repeat_min and max_repeat(ref.split()) < repeat_min,
            "flag_foreign_script": non_arabic_ratio(hyp) > foreign_max and non_arabic_ratio(ref) <= foreign_max}

def _limit_peak(y):
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    return (y / peak if peak > 1.0 else y).astype(np.float32)

def _add_noise_at_snr(y, noise, snr_db):
    scale = _rms(y) / (10 ** (snr_db / 20)) / _rms(noise)
    return _limit_peak(y + noise * scale)

# audio perturbations
def white_noise(snr_db):
    return lambda y, rng: _add_noise_at_snr(y, rng.normal(size=y.shape).astype(np.float32), snr_db)

def real_noise(bank, snr_db):
    def apply(y, rng):
        noise = bank[int(rng.integers(len(bank)))]
        if len(noise) < len(y):
            noise = np.tile(noise, int(np.ceil(len(y) / len(noise))))
        start = int(rng.integers(0, len(noise) - len(y) + 1))
        return _add_noise_at_snr(y, noise[start:start + len(y)], snr_db)
    return apply

# 300-3400 Hz band, 8 kHz, mu-law
def telephone():
    sos = butter(4, [300, 3400], btype="bandpass", fs=SR, output="sos")
    mu = 255.0
    def apply(y, rng):
        y = sosfilt(sos, y).astype(np.float32)
        y = librosa.resample(y, orig_sr=SR, target_sr=8000)
        y = np.clip(y, -1.0, 1.0)
        companded = np.sign(y) * np.log1p(mu * np.abs(y)) / np.log1p(mu)
        companded = np.round(companded * 127) / 127
        y = np.sign(companded) * np.expm1(np.abs(companded) * np.log1p(mu)) / mu
        return librosa.resample(y.astype(np.float32), orig_sr=8000, target_sr=SR).astype(np.float32)
    return apply

def speed(rate):
    return lambda y, rng: librosa.effects.time_stretch(y, rate=rate).astype(np.float32)

def gain(db):
    return lambda y, rng: _limit_peak(y * 10 ** (db / 20))

# synthetic room reverb
def reverb(rt60=0.5):
    def apply(y, rng):
        n = int(SR * rt60)
        t = np.arange(n) / SR
        ir = rng.normal(size=n) * np.exp(-6.9 * t / rt60)
        ir /= np.linalg.norm(ir)
        wet = fftconvolve(y, ir.astype(np.float32))[:len(y)]
        return _limit_peak(wet * (_rms(y) / _rms(wet)))
    return apply

def build_conditions(noise_bank=()):
    conditions = {
        "clean": None,
        "white_20dB": white_noise(20),
        "white_10dB": white_noise(10),
        "white_5dB": white_noise(5),
        "telephone": telephone(),
        "speed_0.85": speed(0.85),
        "speed_1.15": speed(1.15),
        "low_volume_-25dB": gain(-25),
        "reverb_0.5s": reverb(0.5),
    }
    if len(noise_bank):
        for snr in (20, 10, 5):
            conditions[f"real_noise_{snr}dB"] = real_noise(noise_bank, snr)
    return conditions

_AUDIO_COLUMNS = {"audio", "audio_path", "path", "file", "wav_path", "file_name", "filename", "audio_file", "wav"}
_TEXT_COLUMNS = {"transcription", "reference", "text", "ref", "sentence", "ground_truth", "label", "transcript"}
_ID_COLUMNS = {"id", "uid", "utterance_id", "sample_id", "file_id", "name"}
_GENDER_COLUMNS = {"gender", "sex", "speaker_gender"}
_DIALECT_COLUMNS = {"dialect", "accent", "language"}
_AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".webm", ".opus"}

def _find_column(columns, candidates, fallback=None):
    lower_map = {c.lower().strip(): c for c in columns}
    for cand in candidates:
        if cand in lower_map: return lower_map[cand]
    return fallback

def _load_audio(path):
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    return y.astype(np.float32)

def _read_csv_file(path):
    sep = "\t" if Path(path).suffix.lower() == ".tsv" else ","
    return pd.read_csv(path, sep=sep).to_dict("records")

def _read_json_file(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(data, list): return data
    raise ValueError(f"Expected a JSON array at top level in {path}")

def _read_jsonl_file(path):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").strip().splitlines():
        line = line.strip()
        if line: rows.append(json.loads(line))
    return rows

_READERS = {".csv": _read_csv_file, ".tsv": _read_csv_file, ".json": _read_json_file, ".jsonl": _read_jsonl_file}

def _load_from_folder(folder):
    folder = Path(folder)
    audio_files = sorted(p for p in folder.iterdir() if p.suffix.lower() in _AUDIO_EXTS)
    if not audio_files: raise FileNotFoundError(f"No audio files found in {folder}")
    items = []
    for af in audio_files:
        txt_path = af.with_suffix(".txt")
        ref = txt_path.read_text(encoding="utf-8").strip() if txt_path.exists() else ""
        items.append({"audio_path": str(af), "reference_raw": ref, "id": af.stem})
    return items

def _load_from_metadata(meta_path, audio_dir, max_samples, seed):
    meta_path = Path(meta_path)
    reader = _READERS[meta_path.suffix.lower()]
    rows = reader(meta_path)
    if not rows: raise ValueError(f"No rows found in {meta_path}")
    columns = list(rows[0].keys())
    audio_col = _find_column(columns, _AUDIO_COLUMNS)
    text_col = _find_column(columns, _TEXT_COLUMNS)
    id_col = _find_column(columns, _ID_COLUMNS)
    gender_col = _find_column(columns, _GENDER_COLUMNS)
    dialect_col = _find_column(columns, _DIALECT_COLUMNS)
    if text_col is None:
        raise ValueError(f"Cannot find a transcription column in {meta_path}. Available: {columns}")
    if audio_col is None:
        raise ValueError(f"Cannot find an audio path column in {meta_path}. Available: {columns}")
    items = []
    for row in rows:
        items.append({"audio_path": str(row[audio_col]), "reference_raw": str(row[text_col] or ""),
                      "id": str(row.get(id_col, "")) if id_col else "",
                      "gender": str(row.get(gender_col, "unknown")) if gender_col else "unknown",
                      "dialect": str(row.get(dialect_col, "unknown")) if dialect_col else "unknown"})
    return _finalise(items, audio_dir, max_samples, seed)

def _load_from_zip(zip_path, max_samples, seed):
    tmpdir = tempfile.mkdtemp(prefix="stt_eval_")
    with zipfile.ZipFile(zip_path, "r") as zf: zf.extractall(tmpdir)
    extracted = Path(tmpdir)
    meta_files = list(extracted.rglob("*.csv")) + list(extracted.rglob("*.json")) + \
                 list(extracted.rglob("*.jsonl")) + list(extracted.rglob("*.tsv"))
    if meta_files:
        return _load_from_metadata(meta_files[0], extracted, max_samples, seed)
    for folder in [extracted] + list(extracted.iterdir()):
        if folder.is_dir():
            audio_files = [p for p in folder.rglob("*") if p.suffix.lower() in _AUDIO_EXTS]
            if audio_files:
                return _finalise(_load_from_folder(folder), folder, max_samples, seed)
    raise FileNotFoundError(f"No audio or metadata files found inside {zip_path}")

def _finalise(items, audio_dir, max_samples, seed):
    audio_dir = Path(audio_dir)
    rng = np.random.default_rng(seed)
    if max_samples and max_samples < len(items):
        indices = rng.choice(len(items), size=max_samples, replace=False)
        items = [items[i] for i in sorted(indices)]
    result = []
    for i, item in enumerate(items):
        audio_path_str = item.get("audio_path", "")
        if audio_path_str:
            audio_path = Path(audio_path_str)
            if not audio_path.is_absolute(): audio_path = audio_dir / audio_path
            if not audio_path.exists():
                candidate = audio_dir / audio_path.name
                if candidate.exists(): audio_path = candidate
            y = _load_audio(audio_path)
        elif "audio" in item:
            y = np.asarray(item["audio"], dtype=np.float32)
        else:
            raise FileNotFoundError(f"Cannot find audio for item {i}: {item}")
        result.append({"row_index": i, "id": item.get("id", str(i)),
                       "gender": item.get("gender", "unknown"),
                       "dialect": item.get("dialect", "unknown"),
                       "reference_raw": item.get("reference_raw", ""), "audio": y})
    return result

# load ZIP, folder, or metadata file
def load_custom_data(data_path, audio_dir=None, max_samples=None, seed=42):
    data_path = Path(data_path)
    if data_path.suffix.lower() == ".zip":
        return _load_from_zip(data_path, max_samples, seed)
    if data_path.is_dir():
        meta_files = [p for p in data_path.iterdir() if p.suffix.lower() in _READERS]
        if meta_files:
            return _load_from_metadata(meta_files[0], audio_dir or data_path, max_samples, seed)
        return _finalise(_load_from_folder(data_path), audio_dir or data_path, max_samples, seed)
    if data_path.suffix.lower() in _READERS:
        return _load_from_metadata(data_path, audio_dir or data_path.parent, max_samples, seed)
    raise ValueError(f"Unsupported data_path: {data_path}")

# HF ASR pipeline with FP16 support
def build_asr(model_id, device):
    from transformers import pipeline
    return pipeline("automatic-speech-recognition", model=model_id, device=device, chunk_length_s=30,
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

def transcribe(asr, arrays, batch_size, language):
    kwargs = {"task": "transcribe", **(({"language": language} if language else {}))}
    outputs = asr([{"raw": a, "sampling_rate": SR} for a in arrays], batch_size=batch_size, generate_kwargs=kwargs)
    return [o["text"].strip() for o in outputs]

def evaluate_condition(items, transcribe_fn, perturb, rng, norm_cfg, batch_size):
    hyps_raw, audio_seconds, elapsed = [], 0.0, 0.0
    for start in range(0, len(items), batch_size):
        batch = items[start:start + batch_size]
        arrays = [it["audio"] if perturb is None else perturb(it["audio"], rng) for it in batch]
        audio_seconds += sum(len(a) for a in arrays) / SR
        t0 = time.perf_counter()
        hyps_raw.extend(transcribe_fn(arrays))
        elapsed += time.perf_counter() - t0
    refs = [normalize_arabic(it["reference_raw"], norm_cfg) for it in items]
    hyps = [normalize_arabic(h, norm_cfg) for h in hyps_raw]
    scored = score_utterances(refs, hyps)
    keep = [i for i, r in enumerate(refs) if r]
    meta = pd.DataFrame([{k: items[i].get(k, "unknown") for k in ("row_index", "id", "gender", "dialect", "reference_raw")} for i in keep])
    feats = pd.DataFrame([audio_features(items[i]["audio"]) for i in keep])
    flag_df = pd.DataFrame([failure_flags(refs[i], hyps[i]) for i in keep])
    df = pd.concat([meta, feats, scored.reset_index(drop=True), flag_df], axis=1)
    df["prediction_raw"] = [hyps_raw[i] for i in keep]
    return df, elapsed / audio_seconds

# evaluate all noise conditions
def run_conditions(items, transcribe_fn, conditions, norm_cfg, batch_size, seed, n_boot):
    summaries, details = [], {}
    for name in tqdm(list(conditions), desc="Evaluating conditions"):
        rng = np.random.default_rng(seed)
        perturb = conditions[name]
        df, rtf = evaluate_condition(items, transcribe_fn, perturb, rng, norm_cfg, batch_size)
        flag_cols = [c for c in df.columns if c.startswith("flag_")]
        summaries.append({"condition": name, **summarize(df, n_boot, seed), "rtf": rtf,
                          **{f"{c}_rate": float(df[c].mean()) for c in flag_cols},
                          "any_flag_rate": float(df[flag_cols].any(axis=1).mean())})
        details[name] = df
    summary = pd.DataFrame(summaries)
    base = summary.loc[summary.condition == "clean", "wer"]
    if len(base):
        summary["wer_delta_vs_clean"] = summary["wer"] - float(base.iloc[0])
    return summary, details

class EvalResult:
    def __init__(self, summary, utterances, model_name, n_items):
        self.summary = summary
        self.utterances = utterances
        self.model_name = model_name
        self.n_items = n_items
    @property
    def wer(self): return self.summary["wer"]
    @property
    def cer(self): return self.summary["cer"]
    @property
    def rtf(self): return self.summary.get("rtf", 0.0)
    @property
    def worst(self):
        cols = [c for c in ["row_index", "reference_raw", "prediction_raw", "wer"] if c in self.utterances.columns]
        return self.utterances.nlargest(30, "wer")[cols]
    @property
    def confusions(self):
        return top_confusions(self.utterances.ref, self.utterances.hyp)
    def flag_summary(self):
        flag_cols = [c for c in self.utterances.columns if c.startswith("flag_")]
        return {c: float(self.utterances[c].mean()) for c in flag_cols}
    def __repr__(self):
        return f"EvalResult(model={self.model_name!r}, n={self.n_items}, WER={self.wer:.2%}, CER={self.cer:.2%})"

# clean evaluation
def evaluate_model(items, model="openai/whisper-small", language="arabic",
                   batch_size=16, device=None, norm_cfg=NormConfig(), n_boot=1000, seed=42):
    if callable(model):
        transcribe_fn = model
        model_name = getattr(model, "__name__", "custom_model")
    else:
        model_name = str(model)
        if device is None:
            device = 0 if torch.cuda.is_available() else -1
        asr = build_asr(model_name, device)
        transcribe_fn = lambda arrays: transcribe(asr, arrays, batch_size, language)
    hyps_raw, audio_seconds, elapsed = [], 0.0, 0.0
    for start in tqdm(range(0, len(items), batch_size), desc="Transcribing", unit="batch"):
        batch = items[start:start + batch_size]
        arrays = [it["audio"] for it in batch]
        audio_seconds += sum(len(a) for a in arrays) / SR
        t0 = time.perf_counter()
        hyps_raw.extend(transcribe_fn(arrays))
        elapsed += time.perf_counter() - t0
    rtf = elapsed / audio_seconds if audio_seconds > 0 else 0.0
    refs = [normalize_arabic(it["reference_raw"], norm_cfg) for it in items]
    hyps = [normalize_arabic(h, norm_cfg) for h in hyps_raw]
    scored = score_utterances(refs, hyps)
    keep = [i for i, r in enumerate(refs) if r]
    meta = pd.DataFrame([{k: items[i].get(k, "unknown") for k in ("row_index", "id", "gender", "dialect", "reference_raw")} for i in keep])
    feats = pd.DataFrame([audio_features(items[i]["audio"]) for i in keep])
    flag_df = pd.DataFrame([failure_flags(refs[i], hyps[i]) for i in keep])
    df = pd.concat([meta, feats, scored.reset_index(drop=True), flag_df], axis=1)
    df["prediction_raw"] = [hyps_raw[i] for i in keep]
    summary_dict = {**summarize(df, n_boot, seed), "rtf": rtf}
    return EvalResult(summary=summary_dict, utterances=df, model_name=model_name, n_items=len(items))

# WER/CER per dialect
def dialect_summary(df):
    g = df.groupby("dialect")
    out = pd.DataFrame({"n": g.size(),
                        "wer": g.w_err.sum() / g.w_ref_len.sum(),
                        "cer": g.c_err.sum() / g.c_ref_len.sum()})
    return out.sort_values("wer").reset_index()

print("Tools loaded")

---
## 1️⃣ Model & Dataset Configuration

In [ ]:
MODEL = "openai/whisper-large-v3-turbo"
LANGUAGE = "arabic"
BATCH_SIZE = 16

# Options: "hf", "upload_zip", "upload_files", "local", "fleurs"
DATA_SOURCE = "hf"

# 0 = all samples
MAX_SAMPLES = 100

# audio column name in the HF dataset
AUDIO_COL = "audio"

# shuffle stream so dialects are mixed
SHUFFLE = True

# only for DATA_SOURCE = "local"
# DATA_PATH = "/content/drive/MyDrive/my_data/metadata.csv"
# AUDIO_DIR = "/content/drive/MyDrive/my_data/wavs"

print("Model:", MODEL)
print("Language:", LANGUAGE)
print("Batch size:", BATCH_SIZE)
print("Data source:", DATA_SOURCE)
print("Max samples:", MAX_SAMPLES if MAX_SAMPLES else "all")

# optional: custom model, uncomment and edit
# def my_custom_transcribe(audio_arrays):
#     # input: list of 16 kHz mono float32 arrays, output: list of strings
#     return [your_model.predict(a, sampling_rate=16000) for a in audio_arrays]
#
# MODEL = my_custom_transcribe

# optional: mount Google Drive
# from google.colab import drive
# drive.mount("/content/drive")

---
## 2️⃣ Data Loading & Dialectal Streaming

In [ ]:
if DATA_SOURCE == "hf":
    from datasets import Audio as DsAudio, load_dataset

    ds = load_dataset("oddadmix/dialectal-arabic-lahgtna-v2", split="train", streaming=True)

    sample = next(iter(ds))
    print(sample["language"], "-", sample["transcript_text"])

    if SHUFFLE:
        ds = ds.shuffle(seed=42, buffer_size=1000)
    ds = ds.cast_column(AUDIO_COL, DsAudio(sampling_rate=SR))

    limit = MAX_SAMPLES or None
    items = []
    for i, row in enumerate(tqdm(islice(ds, limit), total=limit, desc="Streaming")):
        y = np.asarray(row[AUDIO_COL]["array"], dtype=np.float32)
        if y.ndim == 2: y = y.mean(axis=0 if y.shape[0] <= 8 else 1)
        items.append({"row_index": i, "id": str(i), "gender": "unknown", "dialect": row["language"],
                      "reference_raw": row["transcript_text"], "audio": y})
    print(f"Loaded {len(items)} samples from HF dataset")

elif DATA_SOURCE == "upload_zip":
    from google.colab import files
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    items = load_custom_data(zip_name, max_samples=MAX_SAMPLES or None, seed=42)
    print(f"Loaded {len(items)} samples from {zip_name}")

elif DATA_SOURCE == "upload_files":
    from google.colab import files
    print("Upload metadata file (CSV/JSON):")
    uploaded_meta = files.upload()
    meta_name = list(uploaded_meta.keys())[0]
    print("Upload audio files:")
    uploaded_audio = files.upload()
    audio_tmp = Path("uploaded_audio")
    audio_tmp.mkdir(exist_ok=True)
    for fname, content in uploaded_audio.items():
        (audio_tmp / fname).write_bytes(content)
    items = load_custom_data(meta_name, audio_dir=str(audio_tmp), max_samples=MAX_SAMPLES or None, seed=42)
    print(f"Loaded {len(items)} samples")

elif DATA_SOURCE == "local":
    items = load_custom_data(DATA_PATH, audio_dir=AUDIO_DIR if "AUDIO_DIR" in dir() else None,
                             max_samples=MAX_SAMPLES or None, seed=42)
    print(f"Loaded {len(items)} samples")

elif DATA_SOURCE == "fleurs":
    from datasets import Audio as DsAudio, load_dataset
    ds = load_dataset("google/fleurs", "ar_eg", split="test")
    if MAX_SAMPLES:
        ds = ds.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(ds))))
    ds = ds.cast_column("audio", DsAudio(sampling_rate=SR))
    gender_names = ds.features["gender"].names if hasattr(ds.features["gender"], "names") else None
    items = []
    for i, row in enumerate(ds):
        y = np.asarray(row["audio"]["array"], dtype=np.float32)
        if y.ndim == 2: y = y.mean(axis=0 if y.shape[0] <= 8 else 1)
        gender = gender_names[row["gender"]] if gender_names else row["gender"]
        items.append({"row_index": i, "id": row["id"], "gender": gender, "dialect": "ar_eg",
                      "reference_raw": row["transcription"], "audio": y})
    print(f"Loaded {len(items)} FLEURS samples")

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

preview = pd.DataFrame([{"id": it["id"], "dialect": it.get("dialect", "unknown"),
                         "reference": it["reference_raw"][:80],
                         "duration_sec": round(len(it["audio"]) / SR, 2)} for it in items[:5]])
print("First 5 samples:")
display(preview)

---
## 3️⃣ Baseline Evaluation (Clean Audio)

In [ ]:
import gc
import torch

# Force garbage collection and empty PyTorch's cache to free up GPU memory
gc.collect()
torch.cuda.empty_cache()

with torch.no_grad():
    result = evaluate_model(
        items=items,
        model=MODEL,
        language=LANGUAGE,
        batch_size=BATCH_SIZE,
        norm_cfg=NormConfig(),
        n_boot=1000,
        seed=42,
    )

print("=" * 60)
print("Results for model:", result.model_name)
print("=" * 60)
print(f"  Samples scored:  {result.summary['n_scored']}")
print(f"  WER:             {result.wer:.2%}")
print(f"  CER:             {result.cer:.2%}")
print(f"  RTF:             {result.rtf:.3f}")
print(f"  Substitutions:   {result.summary['sub_rate']:.2%}")
print(f"  Deletions:       {result.summary['del_rate']:.2%}")
print(f"  Insertions:      {result.summary['ins_rate']:.2%}")
print(f"  WER 95% CI:      [{result.summary['wer_ci_low']:.2%}, {result.summary['wer_ci_high']:.2%}]")
print("=" * 60)

---
## 4️⃣ Results Dashboard & Error Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# WER per utterance
axes[0].hist(result.utterances["wer"], bins=30, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].axvline(result.wer, color="red", linestyle="--", linewidth=2, label=f"Corpus WER = {result.wer:.2%}")
axes[0].set_xlabel("WER per utterance")
axes[0].set_ylabel("Count")
axes[0].set_title("WER distribution")
axes[0].legend()

# WER vs duration
axes[1].scatter(result.utterances["duration_sec"], result.utterances["wer"], alpha=0.5, s=20, color="#4CAF50")
axes[1].set_xlabel("Duration (sec)")
axes[1].set_ylabel("WER")
axes[1].set_title("WER vs duration")

# error types
error_types = ["Substitution", "Deletion", "Insertion"]
error_values = [result.summary["sub_rate"], result.summary["del_rate"], result.summary["ins_rate"]]
axes[2].bar(error_types, error_values, color=["#FF9800", "#F44336", "#9C27B0"], edgecolor="white")
axes[2].set_ylabel("Rate")
axes[2].set_title("Error types")
for i, v in enumerate(error_values):
    axes[2].text(i, v + 0.005, f"{v:.2%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 5️⃣ Dialectal Breakdown (WER & CER by Dialect)

In [ ]:
dialect_df = dialect_summary(result.utterances)
print("WER/CER per dialect:")
display(dialect_df.style.format({"wer": "{:.2%}", "cer": "{:.2%}"}))

fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(dialect_df))))
ax.barh(dialect_df["dialect"].astype(str), dialect_df["wer"], color="#2196F3", edgecolor="white")
ax.set_xlabel("WER")
ax.set_title("WER per dialect")
for i, (v, n) in enumerate(zip(dialect_df["wer"], dialect_df["n"])):
    ax.text(v + 0.005, i, f"{v:.1%} (n={n})", va="center", fontsize=8)
plt.tight_layout()
plt.show()

---
## 6️⃣ Error Inspection & Embedded Audio Player

In [ ]:
worst_n = 10
worst_indices = result.utterances.nlargest(worst_n, "wer").index.tolist()

for rank, idx in enumerate(worst_indices, 1):
    row = result.utterances.loc[idx]
    item_idx = int(row["row_index"])
    item = items[item_idx] if item_idx < len(items) else None

    print("-" * 70)
    print(f"#{rank} | WER = {row['wer']:.2%} | duration = {row['duration_sec']:.1f}s | dialect = {row['dialect']}")
    print("Reference:  ", row["reference_raw"])
    print("Prediction: ", row["prediction_raw"])

    if item is not None:
        display(Audio(data=item["audio"], rate=SR))

---
## 7️⃣ Word Confusions Matrix

In [ ]:
confusions = result.confusions
print("Top 25 word errors (sub = substituted, del = dropped, ins = added):")
display(confusions)

---
## 8️⃣ Failure Flags (Hallucinations, Loops & Empty Outputs)

In [ ]:
flags = result.flag_summary()
flag_labels = {
    "flag_empty": "Empty output",
    "flag_too_long": "Output much longer than reference",
    "flag_repetition": "Repetition",
    "flag_foreign_script": "Non-Arabic script",
}

print("Failure flag rates:")
print("-" * 50)
for key, rate in flags.items():
    label = flag_labels.get(key, key)
    bar = "#" * int(rate * 50)
    print(f"  {label}: {rate:.1%}  {bar}")

---
## 9️⃣ Acoustic Robustness & Noise Stress-Testing

In [ ]:
# remove lines to skip conditions
SELECTED_CONDITIONS = {
    "clean": None,
    "white_20dB": white_noise(20),
    "white_10dB": white_noise(10),
    "white_5dB": white_noise(5),
    "telephone": telephone(),
    "speed_0.85": speed(0.85),
    "speed_1.15": speed(1.15),
    "low_volume_-25dB": gain(-25),
    "reverb_0.5s": reverb(0.5),
}

print("Conditions:", list(SELECTED_CONDITIONS.keys()))

if callable(MODEL):
    _transcribe_fn = MODEL
else:
    _device = 0 if torch.cuda.is_available() else -1
    _asr = build_asr(MODEL, _device)
    _transcribe_fn = lambda arrays: transcribe(_asr, arrays, BATCH_SIZE, LANGUAGE)

robustness_summary, robustness_details = run_conditions(
    items=items,
    transcribe_fn=_transcribe_fn,
    conditions=SELECTED_CONDITIONS,
    norm_cfg=NormConfig(),
    batch_size=BATCH_SIZE,
    seed=42,
    n_boot=1000,
)

print("=" * 80)
print("Robustness summary")
print("=" * 80)
display_cols = ["condition", "wer", "cer", "rtf"]
if "wer_delta_vs_clean" in robustness_summary.columns:
    display_cols.append("wer_delta_vs_clean")
summary_display = robustness_summary[display_cols].copy()
summary_display["wer"] = summary_display["wer"].apply(lambda x: f"{x:.2%}")
summary_display["cer"] = summary_display["cer"].apply(lambda x: f"{x:.2%}")
summary_display["rtf"] = summary_display["rtf"].apply(lambda x: f"{x:.3f}")
if "wer_delta_vs_clean" in summary_display.columns:
    summary_display["wer_delta_vs_clean"] = summary_display["wer_delta_vs_clean"].apply(
        lambda x: f"+{x:.2%}" if x > 0 else f"{x:.2%}")
display(summary_display)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

conditions_list = robustness_summary["condition"].tolist()
wer_values = robustness_summary["wer"].tolist()
cer_values = robustness_summary["cer"].tolist()

bar_colors = []
for c in conditions_list:
    if c == "clean": bar_colors.append("#4CAF50")
    elif "white" in c: bar_colors.append("#F44336")
    elif "telephone" in c: bar_colors.append("#FF9800")
    elif "speed" in c: bar_colors.append("#2196F3")
    elif "volume" in c: bar_colors.append("#9C27B0")
    elif "reverb" in c: bar_colors.append("#00BCD4")
    else: bar_colors.append("#607D8B")

for ax, vals, name in [(axes[0], wer_values, "WER"), (axes[1], cer_values, "CER")]:
    ax.bar(range(len(conditions_list)), vals, color=bar_colors, edgecolor="white")
    ax.set_xticks(range(len(conditions_list)))
    ax.set_xticklabels(conditions_list, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel(name)
    ax.set_title(f"{name} per condition")
    for i, v in enumerate(vals):
        ax.text(i, v + 0.005, f"{v:.1%}", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

if "wer_delta_vs_clean" in robustness_summary.columns:
    worst_cond = robustness_summary.loc[robustness_summary["wer"].idxmax()]
    best_cond = robustness_summary.loc[robustness_summary["wer"].idxmin()]
    clean_wer = robustness_summary.loc[robustness_summary.condition == "clean", "wer"].iloc[0]
    print(f"Worst condition: {worst_cond['condition']} (WER = {worst_cond['wer']:.2%})")
    print(f"Best condition: {best_cond['condition']} (WER = {best_cond['wer']:.2%})")
    print(f"Max WER increase vs clean: +{worst_cond['wer'] - clean_wer:.2%}")

---
## 🔊 Listen to Acoustic Perturbations

In [ ]:
SAMPLE_IDX = 0

sample = items[SAMPLE_IDX]
print("Reference:", sample["reference_raw"])
print("Clean audio:")
display(Audio(data=sample["audio"], rate=SR))

for cond_name, perturb_fn in SELECTED_CONDITIONS.items():
    if cond_name == "clean" or perturb_fn is None:
        continue
    perturbed = perturb_fn(sample["audio"].copy(), np.random.default_rng(42))
    print(f"{cond_name}:")
    display(Audio(data=perturbed, rate=SR))

---
## 💾 Export & Download Deliverables

In [ ]:
output_dir = Path("outputs") / result.model_name.split("/")[-1]
output_dir.mkdir(parents=True, exist_ok=True)

utterances_path = output_dir / "utterances.csv"
result.utterances.drop(columns=["ref", "hyp"], errors="ignore").to_csv(utterances_path, index=False, encoding="utf-8-sig")
result.worst.to_csv(output_dir / "worst_30.csv", index=False, encoding="utf-8-sig")
result.confusions.to_csv(output_dir / "top_confusions.csv", index=False, encoding="utf-8-sig")
dialect_summary(result.utterances).to_csv(output_dir / "dialect_wer.csv", index=False, encoding="utf-8-sig")

if "robustness_summary" in dir():
    robustness_summary.to_csv(output_dir / "robustness_summary.csv", index=False, encoding="utf-8-sig")

import platform
report = {
    "model": result.model_name,
    "num_examples": result.n_items,
    "num_scored": result.summary["n_scored"],
    "wer": result.wer,
    "cer": result.cer,
    "rtf": result.rtf,
    "sub_rate": result.summary["sub_rate"],
    "del_rate": result.summary["del_rate"],
    "ins_rate": result.summary["ins_rate"],
    "wer_ci_95": [result.summary["wer_ci_low"], result.summary["wer_ci_high"]],
    "flags": result.flag_summary(),
    "dialects": dialect_summary(result.utterances).to_dict("records"),
    "python": platform.python_version(),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
if "robustness_summary" in dir():
    report["robustness"] = robustness_summary[["condition", "wer", "cer", "rtf"]].to_dict("records")

(output_dir / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved results to:", output_dir)
for f in sorted(output_dir.iterdir()):
    print("  ", f.name)

try:
    from google.colab import files
    for f in sorted(output_dir.iterdir()):
        files.download(str(f))
    print("Download started")
except ImportError:
    print("Not running in Colab, files are in", output_dir)